<a href="https://colab.research.google.com/github/elkins-lab/synth-saxs/blob/main/examples/interactive_tutorials/hiv1_rt_hinge_motion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Detecting Allosteric Hinge Motions: HIV-1 Reverse Transcriptase

Inspired by the structural biology research of the Arnold Lab (Rutgers University), this tutorial demonstrates how to use `synth-saxs` to detect large-scale conformational changes in solution.

HIV-1 Reverse Transcriptase (RT) is a highly flexible enzyme. Its "thumb" domain acts like a hinge, transitioning from an **"open"** unliganded state to a **"closed"** state when bound to a DNA/RNA duplex or certain allosteric inhibitors.

We will compute the SAXS profile and $P(r)$ distance distribution for both the open and closed states to see how this hinge motion manifests in the scattering data.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q synth-saxs biotite matplotlib
else:
    sys.path.append("../../")

In [ ]:
import biotite.database.rcsb as rcsb
import biotite.structure.io as strucio
import matplotlib.pyplot as plt
import numpy as np

from synth_saxs import calculate_p_dist, calculate_saxs_profile, preprocess_structure

## 1. Fetching the Conformational States
We will download two distinct structures of HIV-1 RT from the PDB:
- **1DLO**: Unliganded, "Open" state.
- **1RTD**: DNA-bound, "Closed" state.

In [ ]:
print("Fetching structures from RCSB PDB...")

# Fetch Open State
file_open = rcsb.fetch("1DLO", "pdb", ".")
struct_open = strucio.load_structure(file_open)

# Fetch Closed State (DNA-bound)
file_closed = rcsb.fetch("1RTD", "pdb", ".")
struct_closed = strucio.load_structure(file_closed)

print("Structures loaded successfully.")

## 2. Preprocessing & Preserving Nucleic Acids
Crystal structures often contain water and crystallization buffer (like sulfate or glycerol) which must be removed for SAXS simulation. However, we *want* to keep the DNA in the closed state!

We use `preprocess_structure` to strip the buffer while safely preserving the nucleic acids.

In [ ]:
# Preprocess Open State
clean_open = preprocess_structure(struct_open, keep_nucleic_acids=True)

# Preprocess Closed State (keeps the DNA!)
clean_closed = preprocess_structure(struct_closed, keep_nucleic_acids=True)

print(f"Open state: {len(clean_open)} atoms.")
print(f"Closed state: {len(clean_closed)} atoms.")

## 3. Simulating the SAXS Profiles
We calculate the $I(q)$ curves for both states using the standard Debye formulation.

In [ ]:
print("Simulating SAXS profile for Open State...")
q_open, i_open = calculate_saxs_profile(clean_open, q_max=0.3, n_points=150)

print("Simulating SAXS profile for Closed State...")
q_closed, i_closed = calculate_saxs_profile(clean_closed, q_max=0.3, n_points=150)

## 4. Distance Distribution Analysis ($P(r)$)
While the $I(q)$ curves look similar, the Pair-Distance Distribution $P(r)$ is much more intuitive for detecting a hinge motion. As the thumb closes, the maximum dimension ($D_{max}$) of the complex should shrink, and the distribution of intra-molecular distances should shift inward.

In [ ]:
print("Calculating P(r) distributions...")
r_open, p_open = calculate_p_dist(clean_open, bins=100)
r_closed, p_closed = calculate_p_dist(clean_closed, bins=100)

# Normalize P(r) for direct shape comparison
p_open /= np.max(p_open)
p_closed /= np.max(p_closed)

plt.figure(figsize=(8, 5))
plt.plot(r_open, p_open, label="Open (1DLO)", color="blue", linewidth=2)
plt.plot(r_closed, p_closed, label="Closed (1RTD)", color="red", linewidth=2, linestyle="--")

plt.title("Pair-Distance Distribution P(r) of HIV-1 RT")
plt.xlabel(r"Distance $r$ ($\AA$)")
plt.ylabel("Normalized P(r)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Notice how the tail of the distribution (D_max) for the Closed state shifts inward!")